# Clone-family collapse laboratory — T4 / 16 GB

Our own GPT/TinyStories experiment. Completed T4 results and evidence are documented in README.md. Select a GPU runtime. The first run compares hard routing, 125-step sampled warmup, and gradual exploration decay at 12 experts. All runs retain failures and report hard-routing evaluation. The Python expert loop prioritizes inspection over speed.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable,"-m","pip","install","datasets>=3","tokenizers>=0.20","huggingface_hub","matplotlib","numpy"],check=True)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime."
print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 2**30, "GiB")


In [ ]:
import json,os
from pathlib import Path
root=Path("/content/MoE_Clone_Lab")
root.mkdir(exist_ok=True)
os.chdir(root)
payload = {'src/gpt.py': '"""Small causal decoder-only Transformer and FFN-upcycled sparse MoE."""\nimport copy\nimport math\nfrom dataclasses import dataclass\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n@dataclass\nclass GPTConfig:\n    vocab_size: int = 65\n    context: int = 128\n    layers: int = 4\n    width: int = 128\n    heads: int = 4\n    intermediate: int = 256\n    experts: int = 4\n    top_k: int = 2\n\nclass SwiGLU(nn.Module):\n    def __init__(self, width, intermediate):\n        super().__init__()\n        self.gate = nn.Linear(width, intermediate, bias=False)\n        self.up = nn.Linear(width, intermediate, bias=False)\n        self.down = nn.Linear(intermediate, width, bias=False)\n\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x)) * self.up(x))\n\nclass ExpertFFN(nn.Module):\n    def __init__(self, dense_ffn, width, experts, top_k):\n        super().__init__()\n        if not 1 <= top_k <= experts:\n            raise ValueError(\'Require 1 <= top_k <= experts\')\n        self.top_k = top_k\n        self.experts = nn.ModuleList([copy.deepcopy(dense_ffn) for _ in range(experts)])\n        self.router = nn.Linear(width, experts, bias=False)\n        nn.init.normal_(self.router.weight, std=.02)\n\n    def forward(self, hidden):\n        shape = hidden.shape\n        x = hidden.reshape(-1, shape[-1])\n        # FP32 for ordinary training/autocast, FP64 for independent CPU\n        # reference validation. Do not downcast a double reference gate.\n        gate_dtype = torch.float64 if x.dtype == torch.float64 else torch.float32\n        logits = F.linear(x.to(gate_dtype), self.router.weight.to(gate_dtype))\n        probs = logits.softmax(-1)\n        values, ids = logits.topk(self.top_k, dim=-1)\n        weights = values.softmax(-1).to(x.dtype)\n        output = torch.zeros_like(x)\n        for i, expert in enumerate(self.experts):\n            rows, slots = torch.where(ids == i)\n            if rows.numel():\n                contribution = expert(x[rows]) * weights[rows, slots, None]\n                output = output.index_add(0, rows, contribution)\n        counts = torch.bincount(ids.flatten(), minlength=len(self.experts)).float()\n        fraction = counts / ids.numel()\n        balance = len(self.experts) * (fraction.detach() * probs.mean(0)).sum()\n        entropy = -(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()\n        return output.reshape(shape), {\'balance\': balance, \'counts\': counts, \'entropy\': entropy}\n\nclass Attention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        if cfg.width % cfg.heads:\n            raise ValueError(\'width must be divisible by heads\')\n        self.heads = cfg.heads\n        self.qkv = nn.Linear(cfg.width, 3*cfg.width, bias=False)\n        self.proj = nn.Linear(cfg.width, cfg.width, bias=False)\n\n    def forward(self, x):\n        b,t,d = x.shape\n        q,k,v = self.qkv(x).chunk(3, dim=-1)\n        q,k,v = [a.reshape(b,t,self.heads,d//self.heads).transpose(1,2) for a in (q,k,v)]\n        y = F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.)\n        return self.proj(y.transpose(1,2).contiguous().reshape(b,t,d))\n\nclass Block(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.norm_attn = nn.LayerNorm(cfg.width)\n        self.attn = Attention(cfg)\n        self.norm_ffn = nn.LayerNorm(cfg.width)\n        self.ffn = SwiGLU(cfg.width, cfg.intermediate)\n\n    def forward(self, x):\n        x = x + self.attn(self.norm_attn(x))\n        if isinstance(self.ffn, ExpertFFN):\n            y, aux = self.ffn(self.norm_ffn(x))\n        else:\n            y, aux = self.ffn(self.norm_ffn(x)), None\n        return x+y, aux\n\nclass TinyGPT(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.token_embedding = nn.Embedding(cfg.vocab_size,cfg.width)\n        self.position_embedding = nn.Embedding(cfg.context,cfg.width)\n        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.layers)])\n        self.norm = nn.LayerNorm(cfg.width)\n        self.head = nn.Linear(cfg.width,cfg.vocab_size,bias=False)\n        self.head.weight = self.token_embedding.weight\n        self.apply(self._initialize)\n        for name,p in self.named_parameters():\n            if name.endswith((\'attn.proj.weight\',\'ffn.down.weight\')):\n                nn.init.normal_(p, std=.02/math.sqrt(2*cfg.layers))\n\n    @staticmethod\n    def _initialize(module):\n        if isinstance(module,(nn.Linear,nn.Embedding)):\n            nn.init.normal_(module.weight,std=.02)\n            if isinstance(module,nn.Linear) and module.bias is not None:\n                nn.init.zeros_(module.bias)\n\n    def forward(self, tokens):\n        if tokens.shape[1] > self.cfg.context:\n            raise ValueError(\'Sequence exceeds configured context\')\n        x = self.token_embedding(tokens) + self.position_embedding(torch.arange(tokens.shape[1],device=tokens.device))\n        aux=[]\n        for block in self.blocks:\n            x,a = block(x)\n            if a is not None:\n                aux.append(a)\n        return self.head(self.norm(x)), aux\n\n    def upcycle(self):\n        result = copy.deepcopy(self)\n        if any(isinstance(b.ffn,ExpertFFN) for b in result.blocks):\n            raise ValueError(\'Model is already an MoE\')\n        for block in result.blocks:\n            block.ffn = ExpertFFN(block.ffn,self.cfg.width,self.cfg.experts,self.cfg.top_k).to(self.head.weight.device)\n        return result\n\n    def parameter_counts(self):\n        total=sum(p.numel() for p in self.parameters())\n        inactive=0\n        for block in self.blocks:\n            if isinstance(block.ffn,ExpertFFN):\n                each=sum(p.numel() for p in block.ffn.experts[0].parameters())\n                inactive += (len(block.ffn.experts)-block.ffn.top_k)*each\n        return {\'total\':total, \'active_parameter_convention\':total-inactive,\n                \'convention\':\'All nonexpert parameters, including full tied embedding table, plus selected expert parameters. Not a FLOP count.\'}\n', 'src/routing_lab.py': '"""Controlled clone-family routing experiments; discrete sampled routes, not dense soft MoE."""\nimport copy\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom .gpt import ExpertFFN\n\nclass CloneFFN(ExpertFFN):\n    def __init__(self, parent, copies, router_noise=0., family_routing=False):\n        nn.Module.__init__(self)\n        self.top_k=parent.top_k\n        self.parents=len(parent.experts)\n        self.copies=copies\n        self.family_routing=family_routing\n        self.experts=nn.ModuleList([copy.deepcopy(e) for e in parent.experts for _ in range(copies)])\n        device=parent.router.weight.device\n        self.router=nn.Linear(parent.router.in_features,len(self.experts),bias=False).to(device)\n        with torch.no_grad():\n            self.router.weight.copy_(parent.router.weight.repeat_interleave(copies,dim=0))\n            if router_noise:\n                self.router.weight.add_(torch.randn_like(self.router.weight)*router_noise)\n        self.register_buffer(\'routing_bias\',torch.zeros(len(self.experts),device=device))\n        self.exploration=0.\n        self.temperature=1.\n        self.bias_rate=0.\n        self.route_generator=None\n        self.last_counts=None\n        self.last_hidden=None\n\n    def sample_ids(self,scores,k):\n        # Gumbel-top-k samples WITHOUT replacement. The discrete choice is not differentiated.\n        uniform=torch.rand(scores.shape,device=scores.device,generator=self.route_generator).clamp_(1e-7,1-1e-7)\n        gumbel=-torch.log(-torch.log(uniform))\n        return (scores/self.temperature+gumbel).topk(k,dim=-1).indices\n\n    def select(self,scores):\n        if self.family_routing:\n            family_scores=scores.reshape(-1,self.parents,self.copies).mean(-1)\n            families=family_scores.topk(self.top_k,dim=-1).indices\n            child_scores=scores.reshape(-1,self.parents,self.copies)[torch.arange(len(scores),device=scores.device)[:,None],families]\n            children=child_scores.argmax(-1)\n            hard=families*self.copies+children\n            if self.training and self.exploration:\n                sampled_children=self.sample_ids(child_scores.reshape(-1,self.copies),1).reshape(-1,self.top_k)\n                sampled=families*self.copies+sampled_children\n            else:\n                return hard\n        else:\n            hard=scores.topk(self.top_k,dim=-1).indices\n            if not (self.training and self.exploration):\n                return hard\n            sampled=self.sample_ids(scores,self.top_k)\n        choose=torch.rand((len(scores),1),device=scores.device,generator=self.route_generator)<self.exploration\n        return torch.where(choose,sampled,hard)\n\n    def forward(self,hidden):\n        shape=hidden.shape\n        x=hidden.reshape(-1,shape[-1])\n        logits=F.linear(x.float(),self.router.weight.float())\n        scores=logits+self.routing_bias\n        ids=self.select(scores)\n        # Bias/noise changes selection only; mixing weights use unbiased selected logits.\n        weights=logits.gather(1,ids).softmax(-1).to(x.dtype)\n        output=torch.zeros_like(x)\n        for i,expert in enumerate(self.experts):\n            rows,slots=torch.where(ids==i)\n            if rows.numel():\n                output=output.index_add(0,rows,expert(x[rows])*weights[rows,slots,None])\n        counts=torch.bincount(ids.flatten(),minlength=len(self.experts)).float()\n        probs=logits.softmax(-1)\n        balance=len(self.experts)*((counts/counts.sum()).detach()*probs.mean(0)).sum()\n        self.last_counts=counts.detach()\n        self.last_hidden=x[:32].detach()\n        if self.training and self.bias_rate:\n            with torch.no_grad():\n                self.routing_bias.add_(self.bias_rate*torch.sign(counts.mean()-counts))\n                self.routing_bias.sub_(self.routing_bias.mean())\n        entropy=-(probs*probs.clamp_min(1e-9).log()).sum(-1).mean()\n        return output.reshape(shape),{\'counts\':counts,\'balance\':balance,\'entropy\':entropy}\n\n\ndef grow(model,copies,router_noise=0.,family_routing=False):\n    result=copy.deepcopy(model)\n    for b in result.blocks:\n        if not isinstance(b.ffn,ExpertFFN):\n            raise ValueError(\'Grow an already-trained MoE; dense conversion is a separate phase.\')\n        b.ffn=CloneFFN(b.ffn,copies,router_noise,family_routing)\n    return result\n\n\ndef set_policy(model,exploration,temperature=1.,bias_rate=0.):\n    for b in model.blocks:\n        b.ffn.exploration=exploration\n        b.ffn.temperature=temperature\n        b.ffn.bias_rate=bias_rate\n\n\n@torch.no_grad()\ndef family_output_diversity(ffn):\n    """Relative RMS output distance on the SAME hidden probes; not a switching theorem."""\n    x=ffn.last_hidden\n    if x is None or ffn.copies<2:\n        return []\n    distances=[]\n    for parent in range(ffn.parents):\n        indices=torch.linspace(0,ffn.copies-1,min(ffn.copies,8)).round().long().tolist()\n        outputs=torch.stack([ffn.experts[parent*ffn.copies+j](x) for j in indices]).float()\n        flat=outputs.flatten(1)\n        scale=flat.square().mean().sqrt().clamp_min(1e-8)\n        pair=torch.pdist(flat)/flat.shape[-1]**.5/scale\n        distances.append(pair.mean().item())\n    return distances\n', 'src/train.py': '"""Run from repository root: python -m src.train --device cpu"""\nimport argparse\nimport copy\nimport csv\nimport hashlib\nimport json\nimport platform\nimport subprocess\nimport time\nfrom pathlib import Path\nimport torch\nfrom torch.nn import functional as F\nfrom .models import LinearLM, SparseMoE, parameters\n\n\ndef windows(data, starts, context):\n    return data[starts[:, None] + torch.arange(context)], data[starts + context]\n\n\ndef load_data(path, context):\n    raw = path.read_bytes()\n    text = raw.decode(\'utf-8\')\n    # Split raw text first: no input or target can cross a split boundary.\n    a, b = int(.8 * len(text)), int(.9 * len(text))\n    vocab = sorted(set(text[:a]))\n    mapping = {c: i for i, c in enumerate(vocab)}\n    # This corpus has the same character support in all splits; fail if changed.\n    splits = {k: torch.tensor([mapping[c] for c in t]) for k, t in\n              [(\'train\', text[:a]), (\'validation\', text[a:b]), (\'test\', text[b:])]}\n    manifest = {\'source\': \'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt\',\n                \'sha256\': hashlib.sha256(raw).hexdigest(), \'bytes\': len(raw),\n                \'characters\': len(text), \'vocabulary\': vocab,\n                \'split_characters\': {k: len(v) for k, v in splits.items()},\n                \'split_method\': \'contiguous 80/10/10; split before constructing windows\',\n                \'context\': context}\n    return splits, manifest\n\n\ndef fixed_batch(data, context, n, seed):\n    gen = torch.Generator().manual_seed(seed)\n    starts = torch.randint(len(data) - context, (n,), generator=gen)\n    return windows(data, starts, context)\n\n\n@torch.no_grad()\ndef evaluate(model, batch, device):\n    model.eval()\n    losses, counts, entropies = [], [], []\n    for x, y in zip(batch[0].split(512), batch[1].split(512)):\n        logits, aux = model(x.to(device))\n        losses.append(F.cross_entropy(logits, y.to(device), reduction=\'sum\').item())\n        if aux is not None:\n            counts.append(aux[\'counts\'].cpu())\n            entropies.append((aux[\'entropy\'].item(), len(y)))\n    out = {\'ce\': sum(losses) / len(batch[1])}\n    if counts:\n        load = torch.stack(counts).sum(0)\n        out[\'expert_fraction\'] = (load / load.sum()).tolist()\n        out[\'router_entropy\'] = sum(e*n for e,n in entropies) / len(batch[1])\n    return out\n\n\ndef update(model, optimizer, batch, device, cfg):\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n    logits, aux = model(batch[0].to(device))\n    ce = F.cross_entropy(logits, batch[1].to(device))\n    objective = ce + (cfg[\'balance_coefficient\'] * aux[\'balance\'] if aux else 0)\n    objective.backward()\n    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg[\'gradient_clip\'])\n    if not torch.isfinite(norm) or not torch.isfinite(objective):\n        raise RuntimeError(\'Non-finite gradient or loss\')\n    optimizer.step()\n    return {\'batch_ce\': ce.item(), \'objective\': objective.item(), \'grad_norm\': norm.item()}\n\n\ndef run_phase(model, cfg, splits, evals, device, seed, phase, steps, lr, offset, folder):\n    # Both continuation branches deliberately reset AdamW at conversion.\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg[\'weight_decay\'])\n    gen = torch.Generator().manual_seed(seed + (10000 if phase == \'dense_pretrain\' else 20000))\n    records = []\n    start = time.perf_counter()\n    last = {}\n    for step in range(steps + 1):\n        if step % cfg[\'eval_every\'] == 0 or step == steps:\n            tr = evaluate(model, evals[\'train\'], device)\n            va = evaluate(model, evals[\'validation\'], device)\n            rec = {\'seed\': seed, \'phase\': phase, \'step\': step, \'global_step\': offset + step,\n                   \'targets_seen\': (offset + step) * cfg[\'batch_size\'],\n                   \'train_ce\': tr[\'ce\'], \'validation_ce\': va[\'ce\'],\n                   \'elapsed_seconds_including_eval\': time.perf_counter() - start,\n                   **last}\n            if \'expert_fraction\' in va:\n                rec.update(expert_fraction=va[\'expert_fraction\'], router_entropy=va[\'router_entropy\'])\n            records.append(rec)\n            print(json.dumps(rec), flush=True)\n        if step == steps:\n            break\n        indices = torch.randint(len(splits[\'train\']) - cfg[\'context\'], (cfg[\'batch_size\'],), generator=gen)\n        last = update(model, opt, windows(splits[\'train\'], indices, cfg[\'context\']), device, cfg)\n    if device.startswith(\'cuda\'):\n        torch.cuda.synchronize()\n    elapsed = time.perf_counter() - start\n    checkpoint = {\'model\': model.state_dict(), \'optimizer\': opt.state_dict(),\n                  \'sampler_rng\': gen.get_state(), \'torch_rng\': torch.get_rng_state(),\n                  \'cuda_rng\': torch.cuda.get_rng_state_all() if device.startswith(\'cuda\') else [],\n                  \'phase\': phase, \'step\': steps, \'seed\': seed, \'config\': cfg}\n    checkpoint_path = folder / f\'{phase}.pt\'\n    torch.save(checkpoint, checkpoint_path)\n    # Verify actual disk reload, optimizer restoration, sampler restoration and next update.\n    loaded = torch.load(checkpoint_path, map_location=device, weights_only=True)\n    clone = copy.deepcopy(model)\n    clone.load_state_dict(loaded[\'model\'])\n    clone_opt = torch.optim.AdamW(clone.parameters(), lr=lr, weight_decay=cfg[\'weight_decay\'])\n    clone_opt.load_state_dict(loaded[\'optimizer\'])\n    clone_gen = torch.Generator()\n    clone_gen.set_state(loaded[\'sampler_rng\'].cpu())\n    next_a = torch.randint(len(splits[\'train\']) - cfg[\'context\'], (cfg[\'batch_size\'],), generator=gen)\n    next_b = torch.randint(len(splits[\'train\']) - cfg[\'context\'], (cfg[\'batch_size\'],), generator=clone_gen)\n    assert torch.equal(next_a, next_b)\n    # Work on a copy so verification cannot alter the measured final model.\n    reference = copy.deepcopy(model)\n    ref_opt = torch.optim.AdamW(reference.parameters(), lr=lr, weight_decay=cfg[\'weight_decay\'])\n    ref_opt.load_state_dict(copy.deepcopy(opt.state_dict()))\n    update(reference, ref_opt, windows(splits[\'train\'], next_a, cfg[\'context\']), device, cfg)\n    update(clone, clone_opt, windows(splits[\'train\'], next_b, cfg[\'context\']), device, cfg)\n    resume_error = max((x-y).abs().max().item() for x,y in zip(reference.parameters(), clone.parameters()))\n    assert resume_error < 1e-6, resume_error\n    return records, {\'seconds_including_eval_and_save_excluded\': elapsed,\n                     \'target_tokens_per_second_including_eval\': steps * cfg[\'batch_size\'] / elapsed,\n                     \'resume_next_step_max_parameter_error\': resume_error}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--config\', default=\'configs/routing.json\')\n    parser.add_argument(\'--data\', default=\'data/tinyshakespeare.txt\')\n    parser.add_argument(\'--device\', default=\'cpu\')\n    parser.add_argument(\'--output\', default=\'results/routing\')\n    parser.add_argument(\'--seeds\', type=int, nargs=\'+\')\n    parser.add_argument(\'--threads\', type=int, default=2)\n    args = parser.parse_args()\n    torch.set_num_threads(args.threads)\n    cfg = json.loads(Path(args.config).read_text())\n    if args.seeds:\n        cfg[\'seeds\'] = args.seeds\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=False)  # never overwrite previous evidence\n    cfg_path = out / \'config.json\'\n    cfg_path.write_text(json.dumps(cfg, indent=2))\n    splits, manifest = load_data(Path(args.data), cfg[\'context\'])\n    (out / \'data_manifest.json\').write_text(json.dumps(manifest, indent=2))\n    evals = {k: fixed_batch(v, cfg[\'context\'], cfg[\'eval_samples\'], 9000+i) for i,(k,v) in enumerate(splits.items())}\n    try:\n        commit = subprocess.check_output([\'git\',\'rev-parse\',\'HEAD\'], stderr=subprocess.DEVNULL, text=True).strip()\n    except subprocess.CalledProcessError:\n        commit = None\n    source_hashes = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(Path(\'src\').glob(\'*.py\'))}\n    environment = {\'python\': platform.python_version(), \'torch\': str(torch.__version__),\n                   \'platform\': platform.platform(), \'device\': args.device, \'threads\': args.threads,\n                   \'git_commit\': commit, \'source_sha256\': source_hashes,\n                   \'gpu_name\': torch.cuda.get_device_name() if args.device.startswith(\'cuda\') else None}\n    (out / \'environment.json\').write_text(json.dumps(environment, indent=2))\n    all_records, summaries = [], []\n    for seed in cfg[\'seeds\']:\n        torch.manual_seed(seed)\n        checkpoint_dir = Path(\'checkpoints\') / out.name / str(seed)\n        checkpoint_dir.mkdir(parents=True, exist_ok=True)\n        dense = LinearLM(cfg[\'context\'], len(manifest[\'vocabulary\'])).to(args.device)\n        records, timing = run_phase(dense,cfg,splits,evals,args.device,seed,\'dense_pretrain\',\n                                   cfg[\'dense_steps\'],cfg[\'dense_lr\'],0,checkpoint_dir)\n        all_records += records\n        initial, dense_end = records[0], records[-1]\n        moe = SparseMoE(dense, cfg[\'experts\'], cfg[\'top_k\']).to(args.device)\n        with torch.no_grad():\n            probe = evals[\'validation\'][0].to(args.device)\n            max_error = (dense(probe)[0] - moe(probe)[0]).abs().max().item()\n        assert max_error < 1e-5, max_error\n        torch.save({\'model\':moe.state_dict(),\'config\':cfg,\'seed\':seed},checkpoint_dir/\'moe_converted.pt\')\n        moe_initial_state = copy.deepcopy(moe.state_dict())\n        branches = {}\n        for name, model in [(\'dense_control\', copy.deepcopy(dense)), (\'moe_continue\', moe)]:\n            recs, stats = run_phase(model,cfg,splits,evals,args.device,seed,name,\n                                   cfg[\'continuation_steps\'],cfg[\'continuation_lr\'],cfg[\'dense_steps\'],checkpoint_dir)\n            all_records += recs\n            # Test is measured only at final checkpoints, never used for model selection.\n            branches[name] = {\'start\':recs[0], \'final\':recs[-1], \'test_ce\':evaluate(model,evals[\'test\'],args.device)[\'ce\'], **stats}\n        expert_deltas = [(expert.weight - moe_initial_state[f\'experts.{i}.weight\']).norm().item()\n                         for i,expert in enumerate(moe.experts)]\n        summary = {\'seed\':seed,\'dense_parameters\':parameters(dense),\'moe_total_parameters\':parameters(moe),\n                   \'moe_active_parameters_per_target\':parameters(moe.router)+cfg[\'top_k\']*parameters(dense.linear),\n                   \'conversion_max_logit_error\':max_error,\'initial\':initial,\'dense_end\':dense_end,\n                   \'dense_pretrain_timing\':timing, \'branches\':branches, \'expert_weight_change_l2\':expert_deltas,\n                   \'expert_pair_distance_l2\':(moe.experts[0].weight-moe.experts[1].weight).norm().item()}\n        summary[\'acceptance\'] = {\'dense_train_loss_drops\':dense_end[\'train_ce\'] < initial[\'train_ce\'],\n                                \'dense_validation_loss_drops\':dense_end[\'validation_ce\'] < initial[\'validation_ce\'],\n                                \'moe_train_loss_drops\':branches[\'moe_continue\'][\'final\'][\'train_ce\'] < dense_end[\'train_ce\'],\n                                \'moe_validation_loss_drops\':branches[\'moe_continue\'][\'final\'][\'validation_ce\'] < dense_end[\'validation_ce\'],\n                                \'all_experts_updated\':all(x>0 for x in expert_deltas),\n                                \'conversion_preserved\':max_error<1e-5}\n        summaries.append(summary)\n        (out / \'summary.json\').write_text(json.dumps(summaries, indent=2))\n        with (out / \'metrics.jsonl\').open(\'w\') as f:\n            for rec in all_records:\n                f.write(json.dumps(rec)+\'\\n\')\n    keys = [\'seed\',\'phase\',\'step\',\'global_step\',\'targets_seen\',\'train_ce\',\'validation_ce\',\'elapsed_seconds_including_eval\',\'batch_ce\',\'objective\',\'grad_norm\',\'router_entropy\']\n    with (out / \'metrics.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=keys,extrasaction=\'ignore\'); writer.writeheader(); writer.writerows(all_records)\n    print(\'Acceptance:\', json.dumps([s[\'acceptance\'] for s in summaries]))\n    if not all(all(s[\'acceptance\'].values()) for s in summaries):\n        raise SystemExit(\'One or more acceptance checks failed; preserve evidence and investigate.\')\n\nif __name__ == \'__main__\':\n    main()\n', 'src/clone_sweep.py': '"""Bounded clone-growth lab. Runs only when explicitly invoked; preserves all outcomes."""\nimport argparse,copy,gc,hashlib,json,platform,time\nfrom collections import deque\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom src.gpt import TinyGPT,GPTConfig\nfrom src.routing_lab import grow,set_policy,family_output_diversity\nfrom src.train_gpt import sample,step,evaluate\n\n\ndef counts(model):\n    return torch.stack([b.ffn.last_counts.detach().cpu() for b in model.blocks])\n\n\ndef save(model,opt,gen,path,cfg,extra):\n    torch.save({\'model\':model.state_dict(),\'optimizer\':opt.state_dict(),\'sampler_rng\':gen.get_state(),\n                \'torch_rng\':torch.get_rng_state(),\'cuda_rng\':torch.cuda.get_rng_state_all() if next(model.parameters()).is_cuda else [],\n                \'config\':cfg,**extra},path)\n\n\ndef train_base(model,cfg,data,device,steps,seed):\n    opt=torch.optim.AdamW(model.parameters(),lr=cfg[\'base_lr\'],betas=(.9,.95),weight_decay=.01)\n    gen=torch.Generator().manual_seed(seed)\n    training_cfg={**cfg,\'balance_coefficient\':.01,\'gradient_clip\':1.}\n    for _ in range(steps):\n        step(model,opt,sample(data,gen,cfg[\'batch_size\'],cfg[\'model\'][\'context\']),device,training_cfg)\n    return model\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--config\',default=\'configs/clone_t4.json\')\n    p.add_argument(\'--data\',default=\'data/tinystories_lab.npz\')\n    p.add_argument(\'--device\',default=\'cuda\')\n    p.add_argument(\'--output\',default=\'results/clone-lab\')\n    p.add_argument(\'--arms\',nargs=\'+\',default=[\'hard\',\'warmup\',\'anneal\'])\n    p.add_argument(\'--max-runs\',type=int,default=3)\n    a=p.parse_args()\n    torch.set_num_threads(2)\n    cfg=json.loads(Path(a.config).read_text())\n    out=Path(a.output);out.mkdir(parents=True,exist_ok=False)\n    (out/\'config.json\').write_text(json.dumps(cfg,indent=2))\n    arrays=np.load(a.data)\n    data={k:torch.from_numpy(arrays[k].astype(np.int64)) for k in (\'train\',\'validation\',\'test\')}\n    vocab=int(arrays[\'vocab_size\'])\n    model_cfg=GPTConfig(vocab_size=vocab,**cfg[\'model\'])\n    env={\'torch\':str(torch.__version__),\'python\':platform.python_version(),\'device\':a.device,\n         \'gpu\':torch.cuda.get_device_name() if a.device.startswith(\'cuda\') else None,\n         \'data_sha256\':hashlib.sha256(Path(a.data).read_bytes()).hexdigest(),\n         \'source_sha256\':{str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(\'src\').glob(\'*.py\')}}\n    (out/\'environment.json\').write_text(json.dumps(env,indent=2))\n    checkpoints=Path(\'checkpoints\')/out.name;checkpoints.mkdir(parents=True,exist_ok=True)\n    evals={k:sample(v,torch.Generator().manual_seed(9000+i),cfg[\'eval_sequences\'],model_cfg.context) for i,(k,v) in enumerate(data.items())}\n    calibration=sample(data[\'train\'],torch.Generator().manual_seed(9999),cfg[\'calibration_sequences\'],model_cfg.context)\n    torch.manual_seed(cfg[\'seed\'])\n    base=TinyGPT(model_cfg).to(a.device)\n    base_initial=evaluate(base,evals[\'validation\'],a.device)[\'task_ce\']\n    train_base(base,cfg,data[\'train\'],a.device,cfg[\'dense_steps\'],cfg[\'seed\'])\n    dense_ce=evaluate(base,evals[\'validation\'],a.device)[\'task_ce\']\n    parent=base.upcycle();del base\n    train_base(parent,cfg,data[\'train\'],a.device,cfg[\'parent_steps\'],cfg[\'seed\']+1)\n    parent_ce=evaluate(parent,evals[\'validation\'],a.device)[\'task_ce\']\n    parent_logits=parent.eval()(calibration[0].to(a.device))[0].detach().cpu()\n    base_summary={\'model\':asdict(model_cfg),\'initial_validation_ce\':base_initial,\'dense_validation_ce\':dense_ce,\n                  \'parent_validation_ce\':parent_ce,\'parent_parameters\':parent.parameter_counts(),\n                  \'parent_optimizer\':\'AdamW reset at dense -> parent MoE; balance coefficient .01\'}\n    (out/\'base_summary.json\').write_text(json.dumps(base_summary,indent=2))\n    parent.cpu()\n    torch.save({\'model\':parent.state_dict(),\'model_config\':asdict(model_cfg)},checkpoints/\'parent.pt\')\n    summaries=[];runs=0\n    for expert_count in cfg[\'expert_counts\']:\n        if expert_count%model_cfg.experts:\n            raise ValueError(\'Expert count must be a multiple of original parent count\')\n        for arm in a.arms:\n            if runs>=a.max_runs:\n                (out/\'summary.json\').write_text(json.dumps(summaries,indent=2));return\n            runs+=1\n            name=f\'E{expert_count}-{arm}\'\n            # Fresh independent branches from the SAME CPU parent, not sequentially trained growth.\n            torch.manual_seed(cfg[\'seed\']+2)\n            model=grow(parent,expert_count//model_cfg.experts,\n                       router_noise=cfg.get(\'router_noise\',0.),family_routing=arm==\'family\')\n            params=model.parameter_counts()\n            estimate=params[\'total\']*16\n            result={\'name\':name,\'experts\':expert_count,\'arm\':arm,\'parameters\':params,\n                    \'estimated_parameter_optimizer_bytes\':estimate,\'status\':\'running\'}\n            if a.device.startswith(\'cuda\'):\n                free,_=torch.cuda.mem_get_info()\n                if estimate>free*.75:\n                    result[\'status\']=\'estimated_memory_skip\';summaries.append(result)\n                    (out/\'summary.json\').write_text(json.dumps(summaries,indent=2));del model;continue\n                torch.cuda.reset_peak_memory_stats()\n            opt=None\n            try:\n                model.to(a.device)\n                for b in model.blocks:\n                    b.ffn.route_generator=torch.Generator(device=a.device).manual_seed(cfg[\'seed\']+100)\n                # Conversion drift is measured, NOT assumed to be zero for flat cloned routing.\n                model.eval()\n                with torch.no_grad():\n                    grown_logits=model(calibration[0].to(a.device))[0].cpu()\n                result[\'growth_max_logit_change\']=(grown_logits-parent_logits).abs().max().item()\n                result[\'growth_validation_ce\']=evaluate(model,evals[\'validation\'],a.device)[\'task_ce\']\n                opt=torch.optim.AdamW(model.parameters(),lr=cfg[\'continuation_lr\'],betas=(.9,.95),weight_decay=.01)\n                gen=torch.Generator().manual_seed(cfg[\'seed\']+200)\n                ring=deque(maxlen=cfg[\'window_steps\'])\n                exposure=torch.zeros(model_cfg.layers,expert_count)\n                switch=None;ready_streak=0;hard_probe_ring=deque(maxlen=cfg[\'ready_checks\'])\n                started=time.perf_counter()\n                balance_coefficient=.01 if arm==\'aux\' else 0.\n                train_cfg={**cfg,\'balance_coefficient\':balance_coefficient,\'gradient_clip\':1.}\n                for i in range(cfg[\'steps\']+1):\n                    if arm in (\'hard\',\'aux\',\'bias\'):\n                        exploration=0.\n                    elif arm in (\'warmup\',\'family\'):\n                        exploration=float(i<cfg[\'warmup_steps\'])\n                    elif arm==\'anneal\':\n                        exploration=max(0.,1.-i/cfg[\'anneal_steps\'])\n                    elif arm==\'adaptive\':\n                        exploration=float(switch is None)\n                    else: raise ValueError(\'Unknown arm: \'+arm)\n                    if not exploration and switch is None: switch=i\n                    set_policy(model,exploration,bias_rate=cfg[\'bias_rate\'] if arm==\'bias\' else 0.)\n                    if i%cfg[\'eval_every\']==0 or i==cfg[\'steps\']:\n                        # Evaluation ALWAYS uses hard routing; this does not update routing bias/RNG.\n                        val=evaluate(model,evals[\'validation\'],a.device)\n                        cal=evaluate(model,calibration,a.device)\n                        hard_fractions=torch.tensor(cal[\'expert_fractions\'])\n                        hard_probe_ring.append(hard_fractions)\n                        coverage=torch.stack(list(hard_probe_ring)).sum(0)\n                        hard_zeros=(coverage==0).sum(-1).tolist()\n                        diversity=[family_output_diversity(b.ffn) for b in model.blocks]\n                        training_counts=torch.stack(list(ring)).sum(0) if ring else torch.zeros_like(exposure)\n                        training_zero=(training_counts==0).sum(-1).tolist()\n                        # This is an explicit experimental criterion, not a universal threshold.\n                        ready=(len(hard_probe_ring)==cfg[\'ready_checks\'] and max(hard_zeros)==0\n                               and exposure.min().item()>=cfg[\'min_routings\']\n                               and i>=cfg[\'warmup_steps\'])\n                        if arm==\'adaptive\' and switch is None and ready:\n                            switch=i\n                        record={\'name\':name,\'step\':i,\'exploration\':exploration,\'validation_ce\':val[\'task_ce\'],\n                                \'window_zero_routing_experts\':training_zero,\'hard_probe_zero_experts\':hard_zeros,\n                                \'hard_probe_expert_fractions\':cal[\'expert_fractions\'],\n                                \'minimum_cumulative_routings\':exposure.min().item(),\n                                \'family_relative_output_distance\':diversity,\'switch_step\':switch,\n                                \'elapsed_seconds\':time.perf_counter()-started}\n                        with (out/\'metrics.jsonl\').open(\'a\') as f:f.write(json.dumps(record)+\'\\n\')\n                        print(json.dumps(record),flush=True)\n                        result[\'final\']=record\n                        if time.perf_counter()-started>cfg[\'max_arm_seconds\']:\n                            result[\'status\']=\'time_budget_exhausted\';break\n                    if i==cfg[\'steps\']:\n                        result[\'status\']=\'completed\';break\n                    step(model,opt,sample(data[\'train\'],gen,cfg[\'batch_size\'],model_cfg.context),a.device,train_cfg)\n                    routed=counts(model);ring.append(routed);exposure+=routed\n                result[\'switch_step\']=switch\n                result[\'post_switch_steps\']=max(0,result[\'final\'][\'step\']-switch) if switch is not None else 0\n                result[\'test_ce\']=evaluate(model,evals[\'test\'],a.device)[\'task_ce\']\n                result[\'peak_cuda_allocated_bytes\']=torch.cuda.max_memory_allocated() if a.device.startswith(\'cuda\') else None\n                route_rng=[b.ffn.route_generator.get_state() for b in model.blocks]\n                save(model,opt,gen,checkpoints/f\'{name}.pt\',cfg,{\'expert_count\':expert_count,\'arm\':arm,\n                      \'step\':result[\'final\'][\'step\'],\'switch_step\':switch,\'routing_rng\':route_rng,\n                      \'rolling_counts\':list(ring),\'cumulative_routings\':exposure})\n            except torch.cuda.OutOfMemoryError as exc:\n                result.update(status=\'cuda_oom\',error=str(exc))\n                # Original parent checkpoint and previous logged measurements remain usable.\n            except RuntimeError as exc:\n                result.update(status=\'runtime_failure\',error=str(exc))\n            summaries.append(result)\n            (out/\'summary.json\').write_text(json.dumps(summaries,indent=2))\n            del model,opt;gc.collect()\n            if a.device.startswith(\'cuda\'):torch.cuda.empty_cache()\n\nif __name__==\'__main__\':main()\n', 'src/train_gpt.py': '"""Dense GPT -> FFN-upcycled MoE, with matched-token dense continuation."""\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport platform\nimport subprocess\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport torch\nfrom torch.nn import functional as F\nfrom .gpt import GPTConfig,TinyGPT,ExpertFFN\nfrom .train import load_data\nfrom .conversion_checks import verify_model_conversion\n\n\ndef sample(data,gen,n,context):\n    starts=torch.randint(len(data)-context,(n,),generator=gen)\n    sequence=data[starts[:,None]+torch.arange(context+1)]\n    return sequence[:,:-1],sequence[:,1:]\n\n\ndef step(model,opt,batch,device,cfg):\n    model.train()\n    opt.zero_grad(set_to_none=True)\n    x,y=(t.to(device) for t in batch)\n    logits,aux=model(x)\n    ce=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),y.reshape(-1))\n    balance=torch.stack([a[\'balance\'] for a in aux]).mean() if aux else ce.new_tensor(0.)\n    objective=ce+cfg[\'balance_coefficient\']*balance\n    objective.backward()\n    norm=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg[\'gradient_clip\'])\n    if not torch.isfinite(norm) or not torch.isfinite(objective):\n        raise RuntimeError(\'Non-finite gradients/objective\')\n    router_grad=[b.ffn.router.weight.grad.norm().item() for b in model.blocks if isinstance(b.ffn,ExpertFFN)]\n    opt.step()\n    return {\'batch_task_ce\':ce.item(),\'balance_loss\':balance.item(),\'objective\':objective.item(),\n            \'gradient_norm_before_clip\':norm.item(),\'router_gradient_norm\':router_grad}\n\n\n@torch.no_grad()\ndef evaluate(model,batch,device):\n    model.eval()\n    loss=0.\n    loads=[]\n    entropy=[]\n    for x,y in zip(batch[0].split(8),batch[1].split(8)):\n        logits,aux=model(x.to(device))\n        loss+=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),y.to(device).reshape(-1),reduction=\'sum\').item()\n        if aux:\n            counts=torch.stack([a[\'counts\'] for a in aux]).cpu()\n            if not loads:\n                loads=[torch.zeros_like(counts)]\n                entropy=[torch.zeros(len(aux))]\n            loads[0]+=counts\n            entropy[0]+=torch.tensor([a[\'entropy\'].item() for a in aux])*y.numel()\n    result={\'task_ce\':loss/batch[1].numel()}\n    if loads:\n        fractions=loads[0]/loads[0].sum(-1,keepdim=True)\n        result.update(expert_fractions=fractions.tolist(),\n                      router_entropy=(entropy[0]/batch[1].numel()).tolist(),\n                      expert_load_cv=(fractions.std(-1,unbiased=False)/fractions.mean(-1)).tolist())\n    return result\n\n\ndef phase(model,cfg,data,evals,seed,name,steps,lr,offset,device,checkpoint_dir,log_path):\n    opt=torch.optim.AdamW(model.parameters(),lr=lr,betas=(.9,.95),weight_decay=cfg[\'weight_decay\'])\n    gen=torch.Generator().manual_seed(seed+(10000 if name==\'dense_pretrain\' else 20000))\n    start=time.perf_counter()\n    records=[]\n    last={}\n    for i in range(steps+1):\n        if i%cfg[\'eval_every\']==0 or i==steps:\n            train=evaluate(model,evals[\'train\'],device)\n            validation=evaluate(model,evals[\'validation\'],device)\n            record={\'seed\':seed,\'phase\':name,\'step\':i,\'global_step\':offset+i,\n                    \'targets_seen\':(offset+i)*cfg[\'batch_size\']*cfg[\'model\'][\'context\'],\n                    \'train_ce\':train[\'task_ce\'],\'validation_ce\':validation[\'task_ce\'],\n                    \'elapsed_seconds_including_eval\':time.perf_counter()-start,**last}\n            for k in (\'expert_fractions\',\'router_entropy\',\'expert_load_cv\'):\n                if k in validation:\n                    record[k]=validation[k]\n            records.append(record)\n            with log_path.open(\'a\') as f:\n                f.write(json.dumps(record)+\'\\n\')\n            print(json.dumps(record),flush=True)\n        if i==steps:\n            break\n        last=step(model,opt,sample(data,gen,cfg[\'batch_size\'],cfg[\'model\'][\'context\']),device,cfg)\n    if device.startswith(\'cuda\'):\n        torch.cuda.synchronize()\n    elapsed=time.perf_counter()-start\n    saved={\'model\':model.state_dict(),\'optimizer\':opt.state_dict(),\'sampler_rng\':gen.get_state(),\n           \'torch_rng\':torch.get_rng_state(),\'cuda_rng\':torch.cuda.get_rng_state_all() if device.startswith(\'cuda\') else [],\n           \'config\':cfg,\'seed\':seed,\'phase\':name,\'step\':steps}\n    path=checkpoint_dir/f\'{name}.pt\'\n    torch.save(saved,path)\n    # Check on-disk model/Adam/sampler restoration with an identical next update.\n    loaded=torch.load(path,map_location=device,weights_only=True)\n    restored=copy.deepcopy(model)\n    restored.load_state_dict(loaded[\'model\'])\n    restored_opt=torch.optim.AdamW(restored.parameters(),lr=lr,betas=(.9,.95),weight_decay=cfg[\'weight_decay\'])\n    restored_opt.load_state_dict(loaded[\'optimizer\'])\n    restored_gen=torch.Generator()\n    restored_gen.set_state(loaded[\'sampler_rng\'].cpu())\n    batch=sample(data,gen,cfg[\'batch_size\'],cfg[\'model\'][\'context\'])\n    restored_batch=sample(data,restored_gen,cfg[\'batch_size\'],cfg[\'model\'][\'context\'])\n    assert all(torch.equal(a,b) for a,b in zip(batch,restored_batch))\n    reference=copy.deepcopy(model)\n    ref_opt=torch.optim.AdamW(reference.parameters(),lr=lr,betas=(.9,.95),weight_decay=cfg[\'weight_decay\'])\n    ref_opt.load_state_dict(copy.deepcopy(opt.state_dict()))\n    step(reference,ref_opt,batch,device,cfg)\n    step(restored,restored_opt,restored_batch,device,cfg)\n    error=max((a-b).abs().max().item() for a,b in zip(reference.parameters(),restored.parameters()))\n    assert error<1e-6,error\n    return records,{\'wall_seconds_including_eval\':elapsed,\n                    \'targets_per_second_including_eval\':steps*cfg[\'batch_size\']*cfg[\'model\'][\'context\']/elapsed,\n                    \'resume_next_update_parameter_error\':error}\n\n\n@torch.no_grad()\ndef generate(model,vocabulary,prompt,device,seed,length=200):\n    gen=torch.Generator(device=device).manual_seed(seed)\n    mapping={c:i for i,c in enumerate(vocabulary)}\n    tokens=torch.tensor([[mapping[c] for c in prompt]],device=device)\n    model.eval()\n    for _ in range(length):\n        logits,_=model(tokens[:,-model.cfg.context:])\n        next_token=torch.multinomial((logits[:,-1]/.8).softmax(-1),1,generator=gen)\n        tokens=torch.cat((tokens,next_token),dim=1)\n    return \'\'.join(vocabulary[i] for i in tokens[0].tolist())\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--config\',default=\'configs/gpt_cpu.json\')\n    p.add_argument(\'--data\',default=\'data/tinyshakespeare.txt\')\n    p.add_argument(\'--device\',default=\'cpu\')\n    p.add_argument(\'--output\',default=\'results/gpt-cpu-pilot\')\n    p.add_argument(\'--threads\',type=int,default=2)\n    p.add_argument(\'--resume-dense-run\',help=\'Prior result directory with completed dense phase and checkpoints/<run-name>/<seed>/dense_pretrain.pt\')\n    a=p.parse_args()\n    torch.set_num_threads(a.threads)\n    cfg=json.loads(Path(a.config).read_text())\n    out=Path(a.output)\n    out.mkdir(parents=True,exist_ok=False)\n    (out/\'config.json\').write_text(json.dumps(cfg,indent=2))\n    splits,manifest=load_data(Path(a.data),cfg[\'model\'][\'context\'])\n    manifest[\'objective\']=\'Causal next-character prediction at every position; input/target shifted by one.\'\n    (out/\'data_manifest.json\').write_text(json.dumps(manifest,indent=2))\n    evals={k:sample(v,torch.Generator().manual_seed(9000+i),cfg[\'eval_sequences\'],cfg[\'model\'][\'context\']) for i,(k,v) in enumerate(splits.items())}\n    try:\n        commit=subprocess.check_output([\'git\',\'rev-parse\',\'HEAD\'],stderr=subprocess.DEVNULL,text=True).strip()\n    except subprocess.CalledProcessError:\n        commit=None\n    hashes={str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(\'src\').glob(\'*.py\')}\n    env={\'python\':platform.python_version(),\'torch\':str(torch.__version__),\'device\':a.device,\'threads\':a.threads,\n         \'gpu_name\':torch.cuda.get_device_name() if a.device.startswith(\'cuda\') else None,\n         \'git_commit\':commit,\'source_sha256\':hashes,\'precision\':\'FP32; no autocast/dropout\'}\n    (out/\'environment.json\').write_text(json.dumps(env,indent=2))\n    summaries=[]\n    samples={}\n    for seed in cfg[\'seeds\']:\n        torch.manual_seed(seed)\n        if a.device.startswith(\'cuda\'):\n            torch.cuda.reset_peak_memory_stats()\n        checkpoint_dir=Path(\'checkpoints\')/out.name/str(seed)\n        checkpoint_dir.mkdir(parents=True,exist_ok=True)\n        model=TinyGPT(GPTConfig(vocab_size=len(manifest[\'vocabulary\']),**cfg[\'model\'])).to(a.device)\n        if a.resume_dense_run:\n            source=Path(a.resume_dense_run)\n            source_manifest=json.loads((source/\'data_manifest.json\').read_text())\n            if source_manifest[\'sha256\']!=manifest[\'sha256\']:\n                raise ValueError(\'Resume dataset checksum mismatch\')\n            path=Path(\'checkpoints\')/source.name/str(seed)/\'dense_pretrain.pt\'\n            loaded=torch.load(path,map_location=a.device,weights_only=True)\n            if loaded[\'seed\']!=seed or loaded[\'phase\']!=\'dense_pretrain\' or loaded[\'step\']!=cfg[\'dense_steps\']:\n                raise ValueError(\'Resume checkpoint seed/phase/step mismatch\')\n            for key in (\'model\',\'dense_steps\',\'dense_lr\',\'batch_size\',\'weight_decay\',\'gradient_clip\',\'eval_sequences\'):\n                if loaded[\'config\'][key]!=cfg[key]:\n                    raise ValueError(f\'Resume configuration mismatch: {key}\')\n            model.load_state_dict(loaded[\'model\'])\n            torch.set_rng_state(loaded[\'torch_rng\'].cpu())\n            if a.device.startswith(\'cuda\') and loaded[\'cuda_rng\']:\n                torch.cuda.set_rng_state_all([r.cpu() for r in loaded[\'cuda_rng\']])\n            dense_records=[json.loads(line) for line in (source/\'metrics.jsonl\').read_text().splitlines()]\n            dense_records=[r for r in dense_records if r[\'seed\']==seed and r[\'phase\']==\'dense_pretrain\']\n            if not dense_records or dense_records[0][\'step\']!=0 or dense_records[-1][\'step\']!=cfg[\'dense_steps\']:\n                raise ValueError(\'Resume requires complete dense metrics\')\n            with (out/\'metrics.jsonl\').open(\'a\') as f:\n                for r in dense_records:f.write(json.dumps(r)+\'\\n\')\n            dense_timing={\'wall_seconds_including_eval\':dense_records[-1][\'elapsed_seconds_including_eval\'],\n                          \'resumed_from\':str(source),\'checkpoint_sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\n                          \'resume_next_update_parameter_error\':None,\n                          \'note\':\'Dense timing/metrics inherited; continuation optimizer resets as before.\'}\n            torch.save(loaded,checkpoint_dir/\'dense_pretrain.pt\')\n            print(f\'Restored dense checkpoint: {path}; skipping pretraining\',flush=True)\n        else:\n            dense_records,dense_timing=phase(model,cfg,splits[\'train\'],evals,seed,\'dense_pretrain\',cfg[\'dense_steps\'],cfg[\'dense_lr\'],0,a.device,checkpoint_dir,out/\'metrics.jsonl\')\n        model.eval()\n        moe=model.upcycle().eval()\n        # Exact parameter-copy checks catch structural errors independently of\n        # floating-point kernel differences in sparse subset execution.\n        for dense_block,moe_block in zip(model.blocks,moe.blocks):\n            for expert in moe_block.ffn.experts:\n                for original,copied in zip(dense_block.ffn.parameters(),expert.parameters()):\n                    if not torch.equal(original,copied):\n                        raise RuntimeError(\'Expert weight copy mismatch\')\n        with torch.no_grad():\n            probe,targets=(t.to(a.device) for t in evals[\'validation\'])\n            conversion=verify_model_conversion(model,moe,probe,targets)\n        conversion_error=conversion[\'max_absolute_logit_error\']\n        (out/f\'conversion-{seed}.json\').write_text(json.dumps(conversion,indent=2))\n        print(\'Conversion diagnostics:\',json.dumps(conversion),flush=True)\n        if not conversion[\'passed\']:\n            raise RuntimeError(f\'Conversion check failed; see {out}/conversion-{seed}.json: {conversion}\')\n        torch.save({\'model\':moe.state_dict(),\'config\':cfg,\'seed\':seed},checkpoint_dir/\'moe_converted.pt\')\n        original_experts={k:v.detach().clone() for k,v in moe.state_dict().items() if \'.experts.\' in k}\n        branches={}\n        for name,branch in [(\'dense_control\',copy.deepcopy(model)),(\'moe_continue\',moe)]:\n            records,timing=phase(branch,cfg,splits[\'train\'],evals,seed,name,cfg[\'continuation_steps\'],cfg[\'continuation_lr\'],cfg[\'dense_steps\'],a.device,checkpoint_dir,out/\'metrics.jsonl\')\n            branches[name]={\'start\':records[0],\'final\':records[-1],\'test_ce\':evaluate(branch,evals[\'test\'],a.device)[\'task_ce\'],**timing}\n            samples[f\'{seed}/{name}\']=generate(branch,manifest[\'vocabulary\'],\'First Citizen:\\n\',a.device,1234)\n        deltas={k:(v-original_experts[k]).norm().item() for k,v in moe.state_dict().items() if k in original_experts}\n        all_updated=all(v>0 for v in deltas.values())\n        acceptance={\'dense_train_drops\':dense_records[-1][\'train_ce\']<dense_records[0][\'train_ce\'],\n                    \'dense_validation_drops\':dense_records[-1][\'validation_ce\']<dense_records[0][\'validation_ce\'],\n                    \'moe_train_drops\':branches[\'moe_continue\'][\'final\'][\'train_ce\']<dense_records[-1][\'train_ce\'],\n                    \'moe_validation_drops\':branches[\'moe_continue\'][\'final\'][\'validation_ce\']<dense_records[-1][\'validation_ce\'],\n                    \'all_expert_matrices_updated\':all_updated,\'conversion_preserved\':conversion[\'passed\']}\n        summary={\'seed\':seed,\'model_config\':asdict(model.cfg),\'dense_parameters\':model.parameter_counts(),\n                 \'moe_parameters\':moe.parameter_counts(),\'conversion_max_logit_error\':conversion_error,\'conversion_diagnostics\':conversion,\n                 \'initial\':dense_records[0],\'dense_end\':dense_records[-1],\'dense_timing\':dense_timing,\n                 \'branches\':branches,\'expert_matrix_changes_l2\':deltas,\'acceptance\':acceptance,\n                 \'peak_cuda_memory_allocated_bytes\':torch.cuda.max_memory_allocated() if a.device.startswith(\'cuda\') else None}\n        summaries.append(summary)\n        (out/\'summary.json\').write_text(json.dumps(summaries,indent=2))\n        (out/\'generation_samples.json\').write_text(json.dumps(samples,indent=2))\n    print(\'Acceptance:\',json.dumps([s[\'acceptance\'] for s in summaries]),flush=True)\n    if not all(all(s[\'acceptance\'].values()) for s in summaries):\n        raise SystemExit(\'Acceptance failure; preserve the run and investigate, without overwriting it.\')\n\nif __name__==\'__main__\':\n    main()\n', 'src/models.py': '"""Literal affine predictor and sparse, function-preserving MoE conversion."""\nimport copy\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass LinearLM(nn.Module):\n    def __init__(self, context, vocab):\n        super().__init__()\n        self.context, self.vocab = context, vocab\n        self.linear = nn.Linear(context * vocab, vocab)\n\n    def features(self, tokens):\n        return F.one_hot(tokens, self.vocab).flatten(1).float()\n\n    def forward(self, tokens):\n        return self.linear(self.features(tokens)), None\n\nclass SparseMoE(nn.Module):\n    def __init__(self, dense, experts=4, top_k=2):\n        super().__init__()\n        if not 1 <= top_k <= experts:\n            raise ValueError(\'Require 1 <= top_k <= experts\')\n        self.context, self.vocab = dense.context, dense.vocab\n        self.top_k = top_k\n        self.experts = nn.ModuleList([copy.deepcopy(dense.linear) for _ in range(experts)])\n        self.router = nn.Linear(self.context * self.vocab, experts)\n        nn.init.normal_(self.router.weight, std=0.02)\n        nn.init.zeros_(self.router.bias)\n\n    def forward(self, tokens):\n        x = F.one_hot(tokens, self.vocab).flatten(1).float()\n        router_logits = self.router(x)\n        probs = router_logits.softmax(-1)\n        selected_logits, ids = router_logits.topk(self.top_k, dim=-1)\n        weights = selected_logits.softmax(-1)\n        output = x.new_zeros(x.shape[0], self.vocab)\n        # Actual sparse dispatch: each expert sees only its assigned samples.\n        for i, expert in enumerate(self.experts):\n            rows, slots = torch.where(ids == i)\n            if rows.numel():\n                contribution = expert(x[rows]) * weights[rows, slots, None]\n                output = output.index_add(0, rows, contribution)\n        counts = torch.bincount(ids.flatten(), minlength=len(self.experts)).float()\n        fraction = counts / ids.numel()\n        balance = len(self.experts) * (fraction.detach() * probs.mean(0)).sum()\n        entropy = -(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()\n        return output, {\'balance\': balance, \'counts\': counts, \'entropy\': entropy}\n\ndef parameters(model):\n    return sum(p.numel() for p in model.parameters())\n', 'tests/test_models.py': "import unittest\nimport torch\nfrom src.models import LinearLM, SparseMoE\n\nclass ConversionTests(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(5)\n        torch.set_num_threads(2)\n        self.dense = LinearLM(4, 7)\n        self.x = torch.randint(7, (31, 4))\n\n    def test_conversion_for_different_k(self):\n        for k in (1, 2, 4):\n            moe = SparseMoE(self.dense, 4, k)\n            torch.testing.assert_close(moe(self.x)[0], self.dense(self.x)[0])\n\n    def test_sparse_dispatch_matches_dense_reference(self):\n        moe = SparseMoE(self.dense, 4, 2)\n        with torch.no_grad():\n            for expert in moe.experts:\n                expert.weight.add_(torch.randn_like(expert.weight)*.1)\n        x = self.dense.features(self.x)\n        values, ids = moe.router(x).topk(2, dim=-1)\n        weights = values.softmax(-1)\n        all_outputs = torch.stack([e(x) for e in moe.experts], dim=1)\n        expected = (all_outputs[torch.arange(len(x))[:,None], ids] * weights[:,:,None]).sum(1)\n        actual, aux = moe(self.x)\n        torch.testing.assert_close(actual, expected)\n        self.assertEqual(int(aux['counts'].sum()), 2*len(x))\n        actual.square().mean().backward()\n        self.assertGreater(moe.router.weight.grad.norm().item(), 0)\n\n    def test_identical_experts_initial_router_task_gradient(self):\n        moe = SparseMoE(self.dense, 4, 2)\n        moe(self.x)[0].square().mean().backward()\n        self.assertLess(moe.router.weight.grad.abs().max().item(), 1e-6)\n\n    def test_invalid_k(self):\n        with self.assertRaises(ValueError):\n            SparseMoE(self.dense, 4, 5)\n\nif __name__ == '__main__':\n    unittest.main()\n", 'tests/test_gpt.py': "import unittest\nimport torch\nfrom src.gpt import GPTConfig, TinyGPT, ExpertFFN\n\nclass GPTTests(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(17)\n        torch.set_num_threads(2)\n        self.model=TinyGPT(GPTConfig(vocab_size=13,context=12,layers=2,width=16,heads=4,intermediate=32))\n        self.tokens=torch.randint(13,(3,12))\n\n    def test_conversion_preserves_full_gpt_logits(self):\n        converted=self.model.upcycle()\n        torch.testing.assert_close(self.model(self.tokens)[0],converted(self.tokens)[0],atol=1e-6,rtol=1e-5)\n        self.assertEqual(len(converted(self.tokens)[1]),2)\n        self.assertIs(converted.head.weight,converted.token_embedding.weight)\n\n    def test_future_token_does_not_change_past_logits(self):\n        for model in (self.model,self.model.upcycle()):\n            changed=self.tokens.clone()\n            changed[:,7:] = (changed[:,7:]+1)%13\n            torch.testing.assert_close(model(self.tokens)[0][:,:7],model(changed)[0][:,:7])\n\n    def test_sparse_ffn_reference_and_gradients(self):\n        moe=self.model.upcycle().blocks[0].ffn\n        with torch.no_grad():\n            moe.experts[0].down.weight.add_(.1)\n        x=torch.randn(3,12,16,requires_grad=True)\n        flat=x.reshape(-1,16)\n        values,ids=moe.router(flat).topk(2,dim=-1)\n        all_outputs=torch.stack([e(flat) for e in moe.experts],dim=1)\n        ref=(all_outputs[torch.arange(len(flat))[:,None],ids]*values.softmax(-1)[:,:,None]).sum(1)\n        actual,aux=moe(x)\n        torch.testing.assert_close(actual.reshape(-1,16),ref)\n        self.assertEqual(int(aux['counts'].sum()),2*len(flat))\n        actual.square().mean().backward()\n        self.assertGreater(moe.router.weight.grad.norm().item(),0)\n\n    def test_parameter_accounting(self):\n        counts=self.model.upcycle().parameter_counts()\n        self.assertGreater(counts['total'],counts['active_parameter_convention'])\n        self.assertGreater(counts['active_parameter_convention'],self.model.parameter_counts()['total'])\n\nif __name__=='__main__':\n    unittest.main()\n", 'tests/test_routing_lab.py': "import unittest\nimport torch\nfrom src.gpt import TinyGPT,GPTConfig\nfrom src.routing_lab import grow,set_policy\n\nclass RoutingLabTests(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(7)\n        torch.set_num_threads(2)\n        self.parent=TinyGPT(GPTConfig(vocab_size=13,context=8,layers=1,width=16,heads=4,intermediate=32,experts=8,top_k=2)).upcycle()\n        self.tokens=torch.randint(13,(4,8))\n\n    def test_cloned_router_rows_and_family_preservation(self):\n        # Parent experts distinct: family-aware routing must preserve the parent function.\n        with torch.no_grad():\n            for i,e in enumerate(self.parent.blocks[0].ffn.experts):\n                e.down.weight.add_(i*.01)\n        child=grow(self.parent,4,family_routing=True).eval()\n        expected=self.parent.eval()(self.tokens)[0]\n        torch.testing.assert_close(child(self.tokens)[0],expected,atol=1e-6,rtol=1e-5)\n        self.assertTrue(torch.equal(child.blocks[0].ffn.router.weight[0],child.blocks[0].ffn.router.weight[1]))\n\n    def test_sampling_uses_unique_experts_and_hard_probe_is_rng_neutral(self):\n        child=grow(self.parent,4)\n        ffn=child.blocks[0].ffn\n        ffn.route_generator=torch.Generator().manual_seed(10)\n        child.train();set_policy(child,1.)\n        ids=ffn.select(torch.zeros(2048,32))\n        self.assertTrue(torch.all(ids[:,0]!=ids[:,1]))\n        self.assertEqual(torch.bincount(ids.flatten(),minlength=32).gt(0).sum().item(),32)\n        state=ffn.route_generator.get_state().clone()\n        child.eval();child(self.tokens)\n        self.assertTrue(torch.equal(state,ffn.route_generator.get_state()))\n\n    def test_positive_bias_favors_starved_expert(self):\n        child=grow(self.parent,4)\n        ffn=child.blocks[0].ffn\n        with torch.no_grad():\n            ffn.routing_bias[3]=10.\n        ids=ffn.select(torch.zeros(10,32)+ffn.routing_bias)\n        self.assertTrue(torch.all((ids==3).any(-1)))\n\nif __name__=='__main__': unittest.main()\n", 'configs/clone_t4.json': '{\n  "model": {\n    "context": 128,\n    "layers": 4,\n    "width": 192,\n    "heads": 6,\n    "intermediate": 384,\n    "experts": 4,\n    "top_k": 2\n  },\n  "batch_size": 4,\n  "eval_sequences": 32,\n  "calibration_sequences": 64,\n  "seed": 17,\n  "dense_steps": 500,\n  "parent_steps": 250,\n  "base_lr": 0.0006,\n  "continuation_lr": 0.0003,\n  "expert_counts": [\n    12\n  ],\n  "steps": 750,\n  "warmup_steps": 125,\n  "anneal_steps": 500,\n  "window_steps": 50,\n  "eval_every": 25,\n  "ready_checks": 3,\n  "min_routings": 1000,\n  "bias_rate": 0.001,\n  "router_noise": 0.0,\n  "max_arm_seconds": 600\n}\n', 'scripts/prepare_tinystories_lab.py': '"""Prepare a frozen bounded TinyStories subset and train a BPE on TRAIN stories only."""\nimport argparse,hashlib,json\nfrom pathlib import Path\nimport numpy as np\nfrom datasets import load_dataset\nfrom huggingface_hub import HfApi\nfrom tokenizers import Tokenizer,models,trainers,pre_tokenizers,decoders\n\np=argparse.ArgumentParser()\np.add_argument(\'--output\',default=\'data/tinystories_lab\')\np.add_argument(\'--train-stories\',type=int,default=12000)\np.add_argument(\'--heldout-stories\',type=int,default=1200)\np.add_argument(\'--vocab-size\',type=int,default=4096)\np.add_argument(\'--revision\',default=\'f54c09fd23315a6f9c86f9dc80f725de7d8f9c64\')\np.add_argument(\'--tokenizer\',help=\'Reuse the archived tokenizer for exact token IDs instead of fitting a new one\')\na=p.parse_args()\nroot=Path(a.output);root.mkdir(parents=True,exist_ok=False)\nrevision=a.revision\nsplits={}\nfor key,n in [(\'train\',a.train_stories),(\'validation\',a.heldout_stories*2)]:\n ds=load_dataset(\'roneneldan/TinyStories\',split=key,revision=revision,streaming=True)\n # Bounded shuffled sample; order and revision are recorded, not a globally uniform dataset sample.\n ds=ds.shuffle(seed=17,buffer_size=2000)\n splits[key]=[r[\'text\'] for r in ds.take(n)]\ntrain=splits[\'train\']; held=splits[\'validation\']\n# Remove exact document overlaps across splits; near duplicates are not addressed here.\nseen=set(hashlib.sha256(t.encode()).hexdigest() for t in train)\nvalidation=[];test=[]\nfor i,text in enumerate(held):\n digest=hashlib.sha256(text.encode()).hexdigest()\n if digest in seen:continue\n seen.add(digest)\n (validation if i%2==0 else test).append(text)\nif a.tokenizer:\n tokenizer=Tokenizer.from_file(a.tokenizer)\n if tokenizer.get_vocab_size()!=a.vocab_size or tokenizer.token_to_id(\'[EOS]\') is None:\n  raise ValueError(\'Archived tokenizer vocabulary/special tokens do not match configuration\')\nelse:\n tokenizer=Tokenizer(models.BPE(unk_token=\'[UNK]\'))\n tokenizer.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False)\n tokenizer.decoder=decoders.ByteLevel()\n trainer=trainers.BpeTrainer(vocab_size=a.vocab_size,special_tokens=[\'[UNK]\',\'[EOS]\'],initial_alphabet=pre_tokenizers.ByteLevel.alphabet())\n tokenizer.train_from_iterator(train,trainer=trainer)\nif a.tokenizer:\n import shutil\n shutil.copyfile(a.tokenizer,root/\'tokenizer.json\')\nelse:\n tokenizer.save(str(root/\'tokenizer.json\'))\neos=tokenizer.token_to_id(\'[EOS]\')\nencoded={}\nfor key,stories in [(\'train\',train),(\'validation\',validation),(\'test\',test)]:\n ids=[]\n for text in stories:ids.extend(tokenizer.encode(text).ids+[eos])\n encoded[key]=np.asarray(ids,dtype=np.int32)\nnp.savez(root.with_suffix(\'.npz\'),**encoded,vocab_size=tokenizer.get_vocab_size())\nmanifest={\'dataset\':\'roneneldan/TinyStories\',\'revision\':revision,\'shuffle_seed\':17,\'shuffle_buffer\':2000,\n          \'train_stories\':len(train),\'validation_stories\':len(validation),\'test_stories\':len(test),\n          \'token_counts\':{k:len(v) for k,v in encoded.items()},\'vocab_size\':tokenizer.get_vocab_size(),\n          \'tokenizer_sha256\':hashlib.sha256((root/\'tokenizer.json\').read_bytes()).hexdigest(),\n          \'packed_policy\':\'concatenate documents with EOS; causal attention may cross EOS\',\n          \'deduplication\':\'exact cross-split document hashing only; not near-duplicate decontamination\'}\n(root/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\nprint(json.dumps(manifest,indent=2))\n', 'scripts/report_clones.py': 'import argparse,json\nfrom pathlib import Path\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\np=argparse.ArgumentParser();p.add_argument(\'--results\',required=True);a=p.parse_args()\nroot=Path(a.results)\nrows=[json.loads(l) for l in (root/\'metrics.jsonl\').read_text().splitlines()] if (root/\'metrics.jsonl\').exists() else []\nsummary=json.loads((root/\'summary.json\').read_text())\nenv=json.loads((root/\'environment.json\').read_text())\nfig,axes=plt.subplots(1,3,figsize=(15,4),layout=\'constrained\')\nfor name in sorted({r[\'name\'] for r in rows}):\n r=[v for v in rows if v[\'name\']==name]\n for ax,key in zip(axes,[\'validation_ce\',\'hard_probe_zero_experts\',\'exploration\']):\n  y=[max(v[key]) if isinstance(v[key],list) else v[key] for v in r]\n  ax.plot([v[\'step\'] for v in r],y,label=name)\nfor ax,title in zip(axes,[\'Hard-routing validation CE\',\'Maximum per-layer hard-probe zero count\',\'Training exploration probability\']):\n ax.set(title=title,xlabel=\'Growth continuation step\');ax.grid(alpha=.2)\naxes[0].legend(fontsize=7)\nfig.savefig(root/\'clone_curves.png\',dpi=160)\nlines=[\'# Clone routing lab outcomes\',\'\',\nf"All validation and calibration probes use HARD routing, even when training is stochastic. Recorded execution: {env[\'device\']} / {env.get(\'gpu\') or \'CPU\'} / PyTorch {env[\'torch\']}.",\'\',\n\'| Arm | Status | Total params | Switch | Post-switch steps | Final hard-probe zero count |\',\n\'|---|---|---:|---:|---:|---|\']\nfor s in summary:\n lines.append(f"| {s[\'name\']} | {s[\'status\']} | {s[\'parameters\'][\'total\']:,} | {s.get(\'switch_step\')} | {s.get(\'post_switch_steps\',0)} | {s.get(\'final\',{}).get(\'hard_probe_zero_experts\')} |")\nlines+=[\'\',\'A zero expert count on one batch is not permanent death. The training-window metric aggregates the last configured number of steps; the hard-probe metric aggregates recent fixed calibration probes. Neither guarantees future survival. Output distances are sampled within clone families and are diagnostics, not a validated switching threshold.\',\'\',\n\'Reported statuses distinguish completed runs from memory skips, CUDA OOM, runtime failures and time limits. A configured but unexecuted size is not a measured outcome.\',\'\',\n\'![Clone curves](clone_curves.png)\',\'\']\n(root/\'REPORT.md\').write_text(\'\\n\'.join(lines))\nprint(\'\\n\'.join(lines[:14]))\n', 'src/conversion_checks.py': '"""Scale-aware FP32 conversion checks with retained behavioral diagnostics."""\nimport copy\nimport torch\nfrom torch.nn import functional as F\n\n\n@torch.no_grad()\ndef conversion_diagnostics(reference, actual, targets, atol=1e-5, rtol=1e-5):\n    if reference.shape != actual.shape:\n        raise ValueError(\'Conversion changed output shape\')\n    difference = (actual-reference).abs()\n    allowed = atol + rtol*reference.abs()\n    finite = bool(torch.isfinite(reference).all() and torch.isfinite(actual).all())\n    dense_ce = F.cross_entropy(reference.reshape(-1,reference.shape[-1]),targets.reshape(-1)).item()\n    moe_ce = F.cross_entropy(actual.reshape(-1,actual.shape[-1]),targets.reshape(-1)).item()\n    close = bool(torch.all(difference <= allowed)) if finite else False\n    ce_close = abs(dense_ce-moe_ce) <= 1e-5\n    return {\'atol\':atol,\'rtol\':rtol,\'ce_atol\':1e-5,\'finite\':finite,\n            \'max_absolute_logit_error\':difference.max().item(),\n            \'rms_logit_error\':difference.square().mean().sqrt().item(),\n            \'max_error_over_allowed\':(difference/allowed).max().item(),\n            \'logits_outside_tolerance\':int((difference>allowed).sum().item()),\n            \'dense_probe_ce\':dense_ce,\'moe_probe_ce\':moe_ce,\'probe_ce_change\':moe_ce-dense_ce,\n            \'max_probability_error\':(actual.softmax(-1)-reference.softmax(-1)).abs().max().item(),\n            \'logits_close\':close,\'ce_close\':ce_close,\'passed\':finite and close and ce_close}\n\n\n@torch.no_grad()\ndef verify_model_conversion(dense, moe, tokens, targets):\n    """Validate full-copy conversion in FP32 and independent FP64 CPU probes.\n\n    Retain the original per-logit FP32 check. If it fails, accept only when\n    a tighter CPU-double reference passes AND explicit FP32 behavioral and\n    error bounds pass. This is not a generic license to ignore mismatches.\n    """\n    stats = conversion_diagnostics(dense(tokens)[0], moe(tokens)[0], targets)\n    indices = torch.linspace(0, len(tokens)-1, min(8, len(tokens))).long()\n    probe = tokens.detach().cpu()[indices]\n    labels = targets.detach().cpu()[indices]\n    dense64 = copy.deepcopy(dense).cpu().double().eval()\n    moe64 = copy.deepcopy(moe).cpu().double().eval()\n    ref = dense64(probe)[0]\n    actual = moe64(probe)[0]\n    difference = (actual-ref).abs()\n    finite64 = bool(torch.isfinite(ref).all() and torch.isfinite(actual).all())\n    close64 = bool(torch.all(difference <= 1e-10 + 1e-10*ref.abs())) if finite64 else False\n    ce_ref = F.cross_entropy(ref.reshape(-1,ref.shape[-1]),labels.reshape(-1)).item()\n    ce_actual = F.cross_entropy(actual.reshape(-1,actual.shape[-1]),labels.reshape(-1)).item()\n    reference_passed = finite64 and close64 and abs(ce_actual-ce_ref) <= 1e-10\n    stats[\'fp64_cpu_reference\'] = {\n        \'sequence_indices\':indices.tolist(),\'atol\':1e-10,\'rtol\':1e-10,\'ce_atol\':1e-10,\n        \'max_absolute_logit_error\':difference.max().item(),\n        \'probe_ce_change\':ce_actual-ce_ref,\'passed\':reference_passed}\n    bounds = {\'max_absolute_logit_error\':1e-4, \'rms_logit_error\':1e-5,\n              \'max_probability_error\':1e-5, \'absolute_probe_ce_change\':1e-5}\n    bounded = (stats[\'finite\'] and stats[\'max_absolute_logit_error\'] <= bounds[\'max_absolute_logit_error\']\n               and stats[\'rms_logit_error\'] <= bounds[\'rms_logit_error\']\n               and stats[\'max_probability_error\'] <= bounds[\'max_probability_error\']\n               and abs(stats[\'probe_ce_change\']) <= bounds[\'absolute_probe_ce_change\'])\n    original_passed = stats[\'passed\']\n    stats[\'fp32_behavior_bounds\'] = bounds\n    stats[\'fp32_behavior_bounds_passed\'] = bounded\n    stats[\'passed\'] = reference_passed and (original_passed or bounded)\n    stats[\'acceptance_method\'] = (\'fp32_per_logit_and_fp64_reference\' if original_passed else\n                                  \'fp64_reference_and_bounded_fp32\') if stats[\'passed\'] else \'rejected\'\n    return stats\n'}
for name,source in payload.items():
    path=Path(name); path.parent.mkdir(parents=True,exist_ok=True); path.write_text(source)
subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"],check=True)


## Prepare bounded data
The BPE vocabulary is trained only on training stories. The exact dataset revision, tokenizer and token counts are saved. Official validation stories are divided into validation and test with exact cross-split document deduplication. Near duplicates remain a limitation. EOS-separated document packing allows cross-document attention. Data preparation refuses to overwrite an existing snapshot.

In [ ]:
if not Path("data/tinystories_lab.npz").exists():
    subprocess.run([sys.executable,"scripts/prepare_tinystories_lab.py"],check=True)
print(Path("data/tinystories_lab/manifest.json").read_text())


## First controlled comparison
All three arms start from the same trained four-expert parent checkpoint, reset AdamW, and see identical training samples. The dense model and parent MoE are trained from scratch in this runtime. Hard-probe measurements are made on training calibration data, separately from validation. At evaluation, stochastic routing is OFF. A single seed is a diagnostic run, not a reliable quality comparison.

In [ ]:
from datetime import datetime,timezone
stamp=datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
run_dir="results/clone-first-"+stamp
subprocess.run([sys.executable,"-m","src.clone_sweep","--device","cuda","--output",run_dir,"--arms","hard","warmup","anneal","--max-runs","3"],check=True)
subprocess.run([sys.executable,"scripts/report_clones.py","--results",run_dir],check=True)
from IPython.display import display,Image,Markdown
display(Markdown(Path(run_dir,"REPORT.md").read_text()))
display(Image(filename=str(Path(run_dir,"clone_curves.png"))))


## Preserve evidence
Download configs, source, results, tokenizer manifest and checkpoints. Raw token arrays are excluded because they can be regenerated from the recorded dataset revision and preparation config. Save the archive before a runtime disconnect. Stochastic-run restart equivalence has not yet been tested; saved routing/sampler RNGs are for follow-up recovery work.

In [ ]:
import zipfile
archive=Path("/content/MoE_Clone_Lab-evidence.zip")
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for path in root.rglob("*"):
        if path.is_file() and "__pycache__" not in path.parts and path.suffix!=".npz":
            z.write(path,path.relative_to(root))
from google.colab import files
files.download(str(archive))
